# EcologicalPublicGoodJ

> JAX version of the Ecological Public Good environment.

In [ ]:
#| default_exp Environments/EcologicalPublicGoodJ

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
from pyCRLD.Environments.BaseJ import ebaseJ
from pyCRLD.Utils.HelpersJ import make_variable_vectorJ
from fastcore.utils import *
from fastcore.test import *
from typing import Iterable
import jax.numpy as jnp
import numpy as np

In [ ]:
#| export
class EcologicalPublicGoodJ(ebaseJ):
    """
    Ecological Public Good Environment.
    """

    def __init__(self,
                 N: int,  # number of agents
                 f: Union[float, Iterable],  # public goods synergy factor
                 c: Union[float, Iterable],  # cost of cooperation
                 m: Union[float, Iterable],  # collapse impact
                 qc: Union[float, Iterable],  # collapse leverage/timescale
                 qr: Union[float, Iterable],  # recovery leverage/timescale
                 degraded_choice=False):  # whether agents have a choice at the degraded state
        self.N = N
        self.M = 2
        self.Z = 2

        self.f = make_variable_vectorJ(f, N)
        self.c = make_variable_vectorJ(c, N)
        self.m = make_variable_vectorJ(m, N)
        self.qc = make_variable_vectorJ(qc, N)
        self.qr = make_variable_vectorJ(qr, N)
        self.degraded_choice = degraded_choice

        self.Aset = self.actions()
        self.Sset = self.states()

        self.state = 1
        super().__init__()

In [ ]:
#| export
@patch
def actions(self: EcologicalPublicGoodJ):
    """The action sets"""
    return [['c', 'd'] for _ in range(self.N)]

@patch
def states(self: EcologicalPublicGoodJ):
    """The states set"""
    return ['g', 'p']

In [ ]:
#| export
@patch
def TransitionTensor(self: EcologicalPublicGoodJ):
    """Get the Transition Tensor."""
    dim = np.concatenate(([self.Z], [self.M for _ in range(self.N)], [self.Z]))
    Tsas = np.ones(dim) * (-1)
    for index, _ in np.ndenumerate(Tsas):
        Tsas[index] = self._transition_probability(index[0], index[1:-1], index[-1])
    return jnp.array(Tsas)

In [ ]:
#| export
@patch
def _transition_probability(self: EcologicalPublicGoodJ,
                            s: int,       # the state index
                            jA: Iterable,  # indices for joint actions
                            s_: int        # the next-state index
                            ) -> float:   # transition probability
    """
    Returns the transition probability for current state `s`, joint action `jA`, and next state `s_`.
    """
    transitionprob = 0

    if self.Sset[s] == 'p':  # if we are in the prosperous state
        # determine the defectors
        defcs = np.where(np.array([act[jA[j]] for j, act
                                   in enumerate(self.Aset)]) == 'd')[0]
        # and add up collapse leverages (= per actor collapse probabilites)
        transitionprob = np.sum([float(self.qc[i]) for i in defcs]) / self.N

        if self.Sset[s_] == 'g':  # if we collapsed
            return transitionprob  # that is our transition probability
        else:  # if we didn't collapse
            return 1 - transitionprob  # it is the "inverse" proability

    else:  # if we are in the degraded state
        if self.degraded_choice:  # and if the agents' choice matter
            # determine cooperator
            coops = np.where(np.array([act[jA[j]] for j, act
                                       in enumerate(self.Aset)]) == 'c')[0]
            # and add up recovery leverages (= per actor recovery probabilites)
            transitionprob = np.sum([float(self.qr[i]) for i in coops]) / self.N

        else:  # if the agents do not have a choice
            transitionprob = float(self.qr.mean())  # take the average collapse leverage

        if self.Sset[s_] == 'p':  # if we recovered
            return transitionprob  # that is our transition probability
        else:  # if we didnt' recovery
            return 1 - transitionprob  # it is the "inverse" probability

In [ ]:
#| export
@patch
def RewardTensor(self: EcologicalPublicGoodJ):
    """Get the Reward Tensor R[i,s,a1,...,aN,s']."""
    dim = np.concatenate(([self.N], [self.Z], [self.M for _ in range(self.N)], [self.Z]))
    Risas = np.zeros(dim)
    for index, _ in np.ndenumerate(Risas):
        Risas[index] = self._reward(index[0], index[1], index[2:-1], index[-1])
    return jnp.array(Risas)

In [ ]:
#| export
@patch
def _reward(self: EcologicalPublicGoodJ,
            i: int,       # the agent index
            s: int,       # the state index
            jA: Iterable,  # indices for joint actions
            s_: int        # the next-state index
            ) -> float:   # reward value
    """
    Returns the reward value for agent `i` in current state `s`, under joint action `jA`, when transitioning to next state `s_`.
    """
    if self.Sset[s] == 'g' or self.Sset[s_] == 'g':  # if either current or next state is degraded
        return float(self.m[i])  # the agents receive the collapse impact

    else:  # if current and next state are prosperous
        # determine cooperators
        coops = np.where(np.array([act[jA[j]] for j, act in enumerate(self.Aset)]) == 'c')[0]
        # and sum up reward contributions
        reward = np.sum([float(self.f[i]) * float(self.c[i]) for i in coops]) / self.N

        if self.Aset[i][jA[i]] == 'c':  # if focal player i is a cooperator
            reward -= float(self.c[i])  # subtract the cost of cooperation
        return reward

In [ ]:
#| export
@patch
def id(self: EcologicalPublicGoodJ):
    """
    Returns id string of environment
    """
    # Default
    f = self.f if len(jnp.unique(self.f)) > 1 else self.f[0]
    c = self.c if len(jnp.unique(self.c)) > 1 else self.c[0]
    m = self.m if len(jnp.unique(self.m)) > 1 else self.m[0]
    qc = self.qc if len(jnp.unique(self.qc)) > 1 else self.qc[0]
    qr = self.qr if len(jnp.unique(self.qr)) > 1 else self.qr[0]

    id = f"{self.__class__.__name__}_"+\
        f"{self.N}_{str(f)}_{str(c)}_{str(m)}_{str(qc)}_{str(qr)}"
    if not self.degraded_choice:
        id += "_NoDegChoi"
    else:
        id += "_DegChoi"
    return id

## Tests

In [ ]:
env = EcologicalPublicGoodJ(N=2, f=1.5, c=1.0, m=-2.0, qc=0.4, qr=0.2)
assert env.T.shape == (2, 2, 2, 2)  # Z, M, M, Z
assert env.R.shape == (2, 2, 2, 2, 2)  # N, Z, M, M, Z
assert jnp.allclose(env.T.sum(-1), 1.0)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()